In [4]:
!pip install pandas

In [5]:
import pandas as pd
import re
import itertools
from collections import Counter

CONTEXT_FEATURES = {
    'PPRE': ['pak', 'bapak', 'bu', 'ibu', 'prof', 'dokter', 'gubernur', 'pj', 'presiden', 'pjs', 'plt', 'bang', 'mas', 'si'],
    'OPOS': ['ketua', 'pemerintah', 'menteri', 'pejabat'],
    'LPRE': ['provinsi', 'kota', 'kampung', 'pulau', 'sungai', 'kali'],
    'LLDR': ['gubernur', 'pemprov', 'presiden', 'walikota'],
    'LOPP': ['di', 'ke', 'dari', 'pada', 'arah', 'wilayah', 'daerah']
}

ORGANIZATION_LIST = ['PSI', 'KPU', 'DPR', 'MPR', 'PBB', 'PLN', 'BI', 'PUPR', 'BASARNAS']

def uji_ner_kalimat(text):
    if not str(text).strip():
        return "Tidak ada entitas"

    tokens = re.findall(r'\w+|[^\w\s]', str(text))
    token_features = []

    for t in tokens:
        if t.isupper() and t.isalpha():
            morph = "UpperCase"
        elif t[0].isupper() and t[1:].islower() if len(t) > 1 else t[0].isupper():
            morph = "TitleCase"
        elif t.islower() and t.isalpha():
            morph = "LowerCase"
        elif t.isdigit():
            morph = "Digit"
        else:
            morph = "Other"

        context = "Tidak ada"
        for key, values in CONTEXT_FEATURES.items():
            if t.lower() in values:
                context = key

        t_low = t.lower()
        if t_low in ['jakarta', 'bandung', 'gubernurnya', 'banjir', 'anies', 'gubernur', 'jateng', 'ibukota', 'ikn', 'nusantara', 'ciliwung']:
            pos_tag = "NOUN"
        elif t_low in ['pergi', 'disalahin', 'melihat', 'tangani', 'dari', 'pindah', 'menjabat', 'nyapres']:
            pos_tag = "VERB"
        elif t_low in ['ke', 'di', 'kepada']:
            pos_tag = "PREP"
        elif t_low in ['kemarin', 'sekarang', 'juga', 'sangat', 'klo', 'yg']:
            pos_tag = "ADV"
        elif t_low in ['gede', 'parah', 'baik', 'benar']:
            pos_tag = "ADJ"
        elif t_low in ['bnjir', 'klo', 'gk', 'psi', 'anis']:
            pos_tag = "OOV"
        else:
            pos_tag = "Other"

        token_features.append({
            'string': t,
            'morph': morph,
            'context': context,
            'pos_tag': pos_tag,
            'entity': 'O'
        })

    n = len(token_features)
    for i in range(n):
        t_str = token_features[i]['string']
        t_low = t_str.lower()

        if t_low in ['jakarta', 'bandung', 'jateng', 'bogor', 'bekasi', 'depok', 'aceh', 'surabaya', 'ikn', 'nusantara', 'ciliwung', 'jkt', 'jakut', 'bayam', 'angke']:
            token_features[i]['entity'] = "LOCATION"
            continue

        if t_low in ['anies', 'anis', 'jokowi', 'joko', 'ahok', 'heru', 'prabowo', 'gibran', 'pramono', 'rano', 'karno', 'doel', 'budi']:
            token_features[i]['entity'] = "PERSON"
            continue

        if token_features[i]['morph'] == "TitleCase" and token_features[i]['pos_tag'] == "NOUN":
            if i > 0 and (token_features[i-1]['context'] == "LPRE" or token_features[i-1]['string'].lower() in CONTEXT_FEATURES['LOPP']):
                if t_low not in ['banjir', 'gubernur', 'pak', 'gubenur', 'presiden', 'pj', 'plt', 'pjs']:
                    token_features[i]['entity'] = "LOCATION"
                    continue
            elif (i < n - 1 and token_features[i+1]['string'].lower() == "banjir") or (i > 0 and "banjir" in token_features[i-1]['string'].lower()):
                if t_low not in ['banjir', 'gubernur', 'pak', 'gubenur', 'presiden']:
                    token_features[i]['entity'] = "LOCATION"
                    continue

        if token_features[i]['morph'] == "UpperCase" or t_str in ORGANIZATION_LIST:
            if token_features[i]['pos_tag'] == "OOV" or t_str in ORGANIZATION_LIST:
                token_features[i]['entity'] = "ORGANIZATION"
                continue

        if token_features[i]['entity'] == 'O':
            if token_features[i]['morph'] == "TitleCase" and token_features[i]['pos_tag'] in ["OOV", "NOUN", "Other"]:
                if token_features[i]['context'] != "LLDR" and t_low not in ["gubernur", "gubenur", "banjir", "presiden", "walikota"]:
                    if i > 0 and token_features[i-1]['context'] == 'PPRE':
                        token_features[i]['entity'] = "PERSON"
                    elif i > 0 and token_features[i-1]['string'].lower() in ['bang', 'mas', 'si', 'pak', 'bu']:
                        token_features[i]['entity'] = "PERSON"

    extracted_entities = []
    for tf in token_features:
        if tf['entity'] != 'O':
            extracted_entities.append(f"{tf['string']} ({tf['entity']})")

    return ", ".join(extracted_entities) if extracted_entities else "Tidak ada entitas"


In [6]:
file_path = "/content/Hasil_Preprocessing2.csv"
df = pd.read_csv(file_path, sep=",")

df['Detected_Entities_InNER'] = df['comment_text_display'].apply(uji_ner_kalimat)
df.to_csv("hasil_ner_full_network.csv", index=False)


In [7]:
edges_undirected = []
nodes_undirected = {}

edges_directed = []
nodes_directed = {}

for index, row in df.iterrows():
    entities_str = row['Detected_Entities_InNER']
    author = str(row['author_name']).strip()

    if entities_str == "Tidak ada entitas" or pd.isna(entities_str):
        continue

    entity_items = [e.strip() for e in str(entities_str).split(",")]
    current_doc_entities = []

    for item in entity_items:
        try:
            name = item[:item.rfind('(')].strip()
            ent_type = item[item.rfind('(')+1 : item.rfind(')')].strip()
            current_doc_entities.append(name)

            if name not in nodes_undirected:
                nodes_undirected[name] = ent_type
            if name not in nodes_directed:
                nodes_directed[name] = ent_type
        except:
            pass

    current_doc_entities = list(set(current_doc_entities))

    if len(current_doc_entities) > 1:
        pairs = list(itertools.combinations(current_doc_entities, 2))
        for pair in pairs:
            sorted_pair = tuple(sorted(pair))
            edges_undirected.append(sorted_pair)

    if pd.notna(author) and author != "nan":
        if author not in nodes_directed:
            nodes_directed[author] = "AUTHOR"

        for ent in current_doc_entities:
            edges_directed.append((author, ent))

edge_counts_undirected = Counter(edges_undirected)
edges_data_undirected = []
for (source, target), weight in edge_counts_undirected.items():
    edges_data_undirected.append({
        'Source': source,
        'Target': target,
        'Weight': weight,
        'Type': 'Undirected'
    })
df_edges_undirected = pd.DataFrame(edges_data_undirected)

nodes_data_undirected = []
for name, ent_type in nodes_undirected.items():
    nodes_data_undirected.append({
        'Id': name,
        'Label': name,
        'entitytype': ent_type
    })
df_nodes_undirected = pd.DataFrame(nodes_data_undirected)

df_nodes_undirected.to_csv("nodes_undirected_entity.csv", index=False)
df_edges_undirected.to_csv("edges_undirected_entity.csv", index=False)

edge_counts_directed = Counter(edges_directed)
edges_data_directed = []
for (source, target), weight in edge_counts_directed.items():
    edges_data_directed.append({
        'Source': source,
        'Target': target,
        'Weight': weight,
        'Type': 'Directed'
    })
df_edges_directed = pd.DataFrame(edges_data_directed)

nodes_data_directed = []
for name, ent_type in nodes_directed.items():
    nodes_data_directed.append({
        'Id': name,
        'Label': name,
        'entitytype': ent_type
    })
df_nodes_directed = pd.DataFrame(nodes_data_directed)

df_nodes_directed.to_csv("nodes_directed_author.csv", index=False)
df_edges_directed.to_csv("edges_directed_author.csv", index=False)


In [8]:
!pip install networkx
import networkx as nx
from networkx.algorithms import community

print("=== ANALISIS TOPIK PEMBICARAAN (ENTITAS) ===")
G_ent = nx.Graph()
for (u, v), w in edge_counts_undirected.items():
    G_ent.add_edge(u, v, weight=w)

try:
    comms_ent = community.louvain_communities(G_ent, weight='weight')
except AttributeError:
    comms_ent = community.greedy_modularity_communities(G_ent, weight='weight')

degree_ent = dict(G_ent.degree(weight='weight'))
for i, comm in enumerate(comms_ent):
    if len(comm) > 1:
        sorted_members = sorted(list(comm), key=lambda x: degree_ent[x], reverse=True)
        top_topics = sorted_members[:10]
        size_pct = (len(comm) / len(G_ent.nodes())) * 100
        print(f"\n--- KOMUNITAS TOPIK {i+1} ({size_pct:.1f}% dari total topik) ---")
        print(f"Fokus Bahasan Utama : {', '.join(top_topics)}")
        if len(sorted_members) > 10:
            print(f"Anggota Terkait     : {', '.join(sorted_members[10:20])}{' ...' if len(sorted_members) > 20 else ''}")

print("\n======================================================\n")
print("=== ANALISIS KELOMPOK PENULIS (ECHO CHAMBERS) ===")
G_auth = nx.Graph()
for (author, ent), w in edge_counts_directed.items():
    G_auth.add_edge(author, ent, weight=w)

try:
    comms_auth = community.louvain_communities(G_auth, weight='weight')
except AttributeError:
    comms_auth = community.greedy_modularity_communities(G_auth, weight='weight')

degree_auth = dict(G_auth.degree(weight='weight'))
for i, comm in enumerate(comms_auth):
    authors_in_comm = [n for n in comm if n in nodes_directed and nodes_directed[n] == 'AUTHOR']
    entities_in_comm = [n for n in comm if n in nodes_directed and nodes_directed[n] != 'AUTHOR']

    if len(authors_in_comm) > 0 and len(entities_in_comm) > 0:
        sorted_entities = sorted(entities_in_comm, key=lambda x: degree_auth[x], reverse=True)
        top_entities = sorted_entities[:5]
        print(f"\nKelompok Penulis {i+1} (Berisi {len(authors_in_comm)} akun/penulis)")
        print(f"Mereka dominan membahas: {', '.join(top_entities)}")


=== ANALISIS TOPIK PEMBICARAAN (ENTITAS) ===

--- KOMUNITAS TOPIK 1 (12.9% dari total topik) ---
Fokus Bahasan Utama : Anies, Heru, jokowi, Joko, joko, JOKO, ciliwung, Budi

--- KOMUNITAS TOPIK 2 (19.4% dari total topik) ---
Fokus Bahasan Utama : Jakarta, Anis, Prabowo, Aceh, ikn, Surabaya, Jateng, Gibran, Bandung, Nusantara
Anggota Terkait     : Bogor, bandung

--- KOMUNITAS TOPIK 3 (17.7% dari total topik) ---
Fokus Bahasan Utama : anies, heru, anis, budi, bogor, depok, bekasi, RANO, gibran, PSI
Anggota Terkait     : prabowo

--- KOMUNITAS TOPIK 4 (17.7% dari total topik) ---
Fokus Bahasan Utama : Jokowi, JKT, JAKARTA, Ahok, ANIES, GK, HERU, BUDI, angke, ANIS
Anggota Terkait     : DOEL

--- KOMUNITAS TOPIK 5 (32.3% dari total topik) ---
Fokus Bahasan Utama : jakarta, Rano, karno, Pramono, pramono, rano, IKN, jkt, Doel, doel
Anggota Terkait     : ahok, aceh, nusantara, Eko, IkN, Pram, Karno, Jakut, Dul, DPR


=== ANALISIS KELOMPOK PENULIS (ECHO CHAMBERS) ===

Kelompok Penulis 1 (Beris